# Programmatic extraction harness — Chile climate-finance sources (OEF)

One harness that **fetches the reviewed source pages and extracts structured fund rows with a single reusable prompt**, producing one combined output (`extracted_inventory.csv`) — the programmatic counterpart to the hand-curated per-source `curate.ipynb` notebooks.

**How it works:** for each grouped source URL → fetch page text → apply the shared `EXTRACTION_PROMPT` (an LLM call) → parse JSON rows in the harmonized schema → concatenate. Deterministic HTML parsing is used where a source has clean tables (MMA); the prompt handles the messy pages (energy microsites, SUBDERE/MINVU, GORE, CORFO).

**Honest limits (read me):**
- The LLM call (`MODEL_CALL`) and live HTTP run in *your* environment — plug in your endpoint + key. This notebook ships a deterministic `MODEL_CALL` **stub** so restart-and-run-all works offline and the plumbing/schema are validated; swap it for a real call to extract live.
- CORFO's list and fondos.gob.cl are JavaScript-rendered — those need a headless-browser fetch (`FETCH_JS`), stubbed here.
- This replaces *transcription* with *extraction*; provenance becomes "fetched + prompt", reproducible by re-running.

In [1]:
import json, pandas as pd
SCHEMA=["source_dataset","funder_institution","program_name","eligible_actor","instrument_type",
 "amount_note","status","recurrence","specificity","climate_relevance","gpc_sectors",
 "access_pathway","detail_level","source_url"]

# Grouped sources actually reviewed. method: html_parse (deterministic) | html_llm | js_llm
SOURCES=[
 dict(source_dataset="cl-mma",       institution="Ministerio del Medio Ambiente",
      url="https://fondos.mma.gob.cl/fpa/", method="html_parse"),     # clean tables -> deterministic
 dict(source_dataset="cl-mma",       institution="Ministerio del Medio Ambiente",
      url="https://fondos.mma.gob.cl/fpr/", method="html_parse"),
 dict(source_dataset="cl-minenergia",institution="Min. Energía / AgenciaSE",
      url="https://www.comunaenergetica.cl/sobre-comuna-energetica/", method="html_llm"),
 dict(source_dataset="cl-minenergia",institution="Min. Energía / AgenciaSE",
      url="https://www.casasolar.cl/", method="html_llm"),
 dict(source_dataset="cl-minenergia",institution="Min. Energía",
      url="https://www.chileatiende.gob.cl/fichas/36666-fondo-de-acceso-a-la-energia-fae", method="html_llm"),
 dict(source_dataset="cl-subdere",   institution="SUBDERE",
      url="https://www.subdere.gov.cl/programas", method="html_llm"),
 dict(source_dataset="cl-minvu",     institution="MINVU",
      url="https://www.minvu.gob.cl/beneficios/ciudad/", method="html_llm"),
 dict(source_dataset="cl-gore",      institution="Gobiernos Regionales",
      url="https://www.subdere.gov.cl/content/glosa-03-provisi%C3%B3n-fondo-nacional-de-desarrollo-regional-fndr", method="html_llm"),
 dict(source_dataset="cl-corfo",     institution="CORFO",
      url="https://www.corfo.cl/sites/cpp/programasyconvocatorias", method="js_llm"),  # JS-rendered
]
print(len(SOURCES),"source URLs across",len({s["source_dataset"] for s in SOURCES}),"institutions")

9 source URLs across 6 institutions


## The reusable extraction prompt
This single prompt reproduces the conventions used across all six reviews. It is the 'prompt that does the same thing'.

In [2]:
EXTRACTION_PROMPT = """You extract climate-relevant public FUNDING OPPORTUNITIES from a Chilean institution's web page into JSON.

Return a JSON list; one object per distinct fund / programme / call on the page. If none, return [].
Each object MUST use exactly these keys:
- program_name: official name of the fund/programme/call.
- funder_institution: the public body that funds it.
- eligible_actor: who can apply (e.g. "municipality", "community/citizen org", "household", "indigenous community", "private firm", "NGO"). If it is explicitly NOT municipalities, say so.
- instrument_type: grant | loan | guarantee | blended | technical_assistance | subsidy | equity.
- amount_note: amount/ceiling/rate as stated (keep units: CLP, UF, US$, %); "" if not stated.
- status: open | closed | ongoing | periodic | emerging (as of the page; prefer the close DATE over section labels).
- recurrence: annual | ongoing | sporadic | one-off (how often it reopens; infer from the page/archive).
- specificity: "sector-specific" if it targets a defined climate sector; "broad" if it is a general-purpose fund that fits many sectors (e.g. general municipal infrastructure).
- climate_relevance: "explicit" (named climate/energy/environment line) | "climate-adjacent" (general infra with climate co-benefits) | "indirect".
- gpc_sectors: list from {stationary_energy, transportation, waste, water, afolu, industry, buildings, cross_sector}, mapped from the funded works (NOT the name alone).
- access_pathway: "direct application" | "facilitated-by-city" | "intermediated (via banks/AE)" | other.
- detail_level: "detailed" if the page states amount+eligibility+dates; "index" if only the programme is identified.

Rules: capture FACTS only; do not invent amounts or dates. Annual funds that are currently closed are still recurrence=annual. Mark broad/general-purpose funds specificity=broad. Output ONLY the JSON list.

PAGE_URL: {url}
PAGE_TEXT:
{page_text}
"""
print(EXTRACTION_PROMPT[:300], "...")

You extract climate-relevant public FUNDING OPPORTUNITIES from a Chilean institution's web page into JSON.

Return a JSON list; one object per distinct fund / programme / call on the page. If none, return [].
Each object MUST use exactly these keys:
- program_name: official name of the fund/programm ...


## Fetch + extract plumbing (pluggable). Swap the stubs for your real fetch + LLM.

In [3]:
def FETCH(url):
    """Live fetch — plug in requests in your env. import requests; return requests.get(url, timeout=30).text"""
    raise NotImplementedError("plug in requests.get; this env uses the platform web tool")

def FETCH_JS(url):
    """Headless-browser fetch for JS-rendered pages (CORFO, fondos.gob.cl)."""
    raise NotImplementedError("plug in Playwright/Selenium get rendered text")

def MODEL_CALL(prompt):
    """LLM call returning the JSON string. Plug in Anthropic/OpenAI:
       resp = client.messages.create(model=..., messages=[{"role":"user","content":prompt}]); return resp...text"""
    return DEMO_RESPONSES.get(_current_url, "[]")   # offline stub (see next cell)

def deterministic_parse(html, url, source_dataset):
    """For clean-table sources (MMA) reuse the cl-mma parser approach (BeautifulSoup).
       Stubbed here to defer to the committed cl-mma extraction notebook."""
    return DEMO_RESPONSES_PARSED.get(url, [])

def extract_source(s, offline=True):
    global _current_url; _current_url=s["url"]
    if s["method"]=="html_parse":
        rows=deterministic_parse(None if offline else FETCH(s["url"]), s["url"], s["source_dataset"])
    else:
        text = "(offline demo text)" if offline else (FETCH_JS if s["method"]=="js_llm" else FETCH)(s["url"])
        raw = MODEL_CALL(EXTRACTION_PROMPT.replace("{url}", s["url"]).replace("{page_text}", text))
        rows = json.loads(raw)
    for r in rows:
        r["source_dataset"]=s["source_dataset"]; r["source_url"]=s["url"]
        r.setdefault("funder_institution", s["institution"])
        if isinstance(r.get("gpc_sectors"), list): r["gpc_sectors"]=json.dumps(r["gpc_sectors"], ensure_ascii=False)
    return rows
print("plumbing defined")

plumbing defined


## Offline demo responses (prove the harness end-to-end without a live LLM).
These are sample outputs in the prompt's format; live runs replace them with real `MODEL_CALL` results.

In [4]:
DEMO_RESPONSES = {
 "https://www.comunaenergetica.cl/sobre-comuna-energetica/": json.dumps([{
   "program_name":"Comuna Energética","funder_institution":"Min. Energía / AgenciaSE",
   "eligible_actor":"municipality","instrument_type":"technical_assistance",
   "amount_note":"co-financing via concursable windows","status":"ongoing","recurrence":"ongoing",
   "specificity":"sector-specific","climate_relevance":"explicit","gpc_sectors":["stationary_energy"],
   "access_pathway":"municipality applies","detail_level":"detailed"}]),
 "https://www.subdere.gov.cl/programas": json.dumps([{
   "program_name":"Programa de Mejoramiento Urbano (PMU)","funder_institution":"SUBDERE",
   "eligible_actor":"municipality","instrument_type":"grant","amount_note":"per-project, budget-dependent",
   "status":"ongoing","recurrence":"ongoing","specificity":"broad","climate_relevance":"climate-adjacent",
   "gpc_sectors":["cross_sector"],"access_pathway":"direct application","detail_level":"detailed"}]),
}
DEMO_RESPONSES_PARSED = {
 "https://fondos.mma.gob.cl/fpa/": [{
   "program_name":"FPA 2026 - Proyectos Sustentables Ciudadanos","funder_institution":"Ministerio del Medio Ambiente",
   "eligible_actor":"community/citizen org","instrument_type":"grant","amount_note":"CLP 6,000,000",
   "status":"closed","recurrence":"annual","specificity":"sector-specific","climate_relevance":"explicit",
   "gpc_sectors":["stationary_energy","waste","afolu"],"access_pathway":"direct application (via fondos.gob.cl)","detail_level":"detailed"}],
}
# default empty for the others in this offline demo
for s in SOURCES:
    DEMO_RESPONSES.setdefault(s["url"], "[]")
print("demo responses for", len(DEMO_RESPONSES), "urls")

demo responses for 9 urls


## Run the harness → one combined output.

In [5]:
all_rows=[]
for s in SOURCES:
    all_rows += extract_source(s, offline=True)
inv = pd.DataFrame(all_rows)
for c in SCHEMA:
    if c not in inv.columns: inv[c]=None
inv = inv[SCHEMA]
# validate the plumbing/schema (not coverage completeness — offline demo only returns sample rows)
assert set(SCHEMA).issubset(inv.columns)
assert inv.source_url.str.startswith("http").all()
assert inv.source_dataset.isin({s["source_dataset"] for s in SOURCES}).all()
OFFLINE_RUN = True   # set False when you plug in a live FETCH + MODEL_CALL
if OFFLINE_RUN:
    print(f"DEMO ONLY — {len(inv)} stub rows; this is NOT the 78-row inventory. "
          "Run live (offline=False, real FETCH/MODEL_CALL) to extract, which writes data/extracted_inventory.csv.")
else:
    inv.to_csv("data/extracted_inventory.csv", index=False)
    print("wrote data/extracted_inventory.csv", inv.shape)
inv[["source_dataset","program_name","instrument_type","specificity"]]

DEMO ONLY — 3 stub rows; this is NOT the 78-row inventory. Run live (offline=False, real FETCH/MODEL_CALL) to extract, which writes data/extracted_inventory.csv.


,source_dataset,program_name,instrument_type,specificity
0,cl-mma,FPA 2026 - Proyectos Sustentables Ciudadanos,grant,sector-specific
1,cl-minenergia,Comuna Energética,technical_assistance,sector-specific
2,cl-subdere,Programa de Mejoramiento Urbano (PMU),grant,broad


## Usage (live)

1. Implement `FETCH` (`requests.get`) and, for JS pages, `FETCH_JS` (Playwright/Selenium).
2. Implement `MODEL_CALL` to call your LLM with the prompt and return the JSON string.
3. Set `offline=False` in the run cell.
4. Output `extracted_inventory.csv` is in the harmonized schema — feed it straight into `harmonize.ipynb` (or replace the per-source `curate.ipynb` CSVs).

This is the reproducible, programmatic replacement for hand-curation: re-running re-fetches and re-extracts, so the data is sourced, not typed. MMA stays on the deterministic parser (`cl_mma_fondos_extract_clean.ipynb`); everything else flows through the single prompt above.